# 05 参数优化与过拟合检验

1. 网格搜索 — 全量枚举参数组合
2. 遗传算法 — 大参数空间高效搜索
3. Walk-Forward Analysis — 样本外验证，防止过拟合

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from futuresquant.data.loader import FuturesDataLoader
from futuresquant.backtest.engine import BacktestConfig
from futuresquant.optimize import GridSearchOptimizer, GeneticOptimizer, WalkForwardOptimizer
from futuresquant.strategy.examples import MACrossStrategy, ATRBreakoutStrategy

DATA_DIR = r'I:\stock\FuturesQuant\raw_data\1min_FU'
CACHE_DIR = r'I:\stock\FuturesQuant\cache'
SYMBOL = 'FU2210'
CONFIG = BacktestConfig(symbol=SYMBOL, initial_capital=1_000_000)

loader = FuturesDataLoader(DATA_DIR, cache_dir=CACHE_DIR)
klines_full = loader.load(SYMBOL, start='2022-01-01', end='2022-10-31')

# IS / OOS 手动分割（用于对比演示）
split_date = '2022-07-01'
klines_is  = klines_full[klines_full.index < split_date]
klines_oos = klines_full[klines_full.index >= split_date]
print(f'IS: {len(klines_is):,} bars   OOS: {len(klines_oos):,} bars')

IS: 13,323 bars   OOS: 10,659 bars


## 1. 网格搜索

In [2]:
factory = lambda fast, slow: MACrossStrategy(fast=fast, slow=slow)
param_grid = {
    'fast': [3, 5, 10, 15, 20],
    'slow': [20, 30, 40, 60, 80],
}
constraints = [lambda p: p['fast'] < p['slow']]

grid_opt = GridSearchOptimizer(
    factory, param_grid, CONFIG,
    metric='sharpe', constraints=constraints
)
grid_result = grid_opt.run(klines_is)

print(f'最优参数: {grid_result.best_params}  Sharpe={grid_result.best_score:.4f}')
grid_result.summary(top_n=8)

Output()

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

...

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

最优参数: {'fast': 3, 'slow': 40}  Sharpe=1.9061


,fast,slow,total_return,annual_return,annual_vol,sharpe,sortino,max_drawdown,calmar,start,end
0,3,40,0.024317,0.114264,0.059945,1.9061,1.6547,-0.014805,7.7178,2022-01-04 09:09:00,2022-06-30 22:57:00
1,3,20,0.021130,0.098738,0.061052,1.6173,1.4184,-0.014442,6.8368,2022-01-04 09:09:00,2022-06-30 22:57:00
2,3,60,0.016696,0.077416,0.060773,1.2739,1.0750,-0.022409,3.4546,2022-01-04 09:09:00,2022-06-30 22:57:00
3,20,60,0.012702,0.058485,0.057196,1.0225,0.9557,-0.010198,5.7350,2022-01-04 09:09:00,2022-06-30 22:57:00
4,3,30,0.012784,0.058869,0.060560,0.9721,0.8594,-0.019301,3.0500,2022-01-04 09:09:00,2022-06-30 22:57:00
5,20,30,0.011308,0.051942,0.057913,0.8969,0.8422,-0.013470,3.8561,2022-01-04 09:09:00,2022-06-30 22:57:00
6,15,40,0.010890,0.049983,0.057603,0.8677,0.8018,-0.012272,4.0728,2022-01-04 09:09:00,2022-06-30 22:57:00
7,5,60,0.011110,0.051013,0.060049,0.8495,0.7243,-0.021914,2.3279,2022-01-04 09:09:00,2022-06-30 22:57:00


In [3]:
# Sharpe 热图
heatmap = grid_result.heatmap_data('fast', 'slow')
fig = px.imshow(
    heatmap, text_auto='.3f', color_continuous_scale='RdYlGn',
    title='网格搜索 Sharpe 热图（IS 期）',
    labels={'x': 'fast window', 'y': 'slow window', 'color': 'Sharpe'}
)
fig.update_layout(height=380)
fig.show()

## 2. 遗传算法（大参数空间）

In [4]:
# 扩大搜索空间，网格搜索会有 ~100 组合，遗传算法只跑 population×generation 次
large_grid = {
    'fast':  list(range(3, 25, 2)),   # 3,5,7,...,23  → 11 个
    'slow':  list(range(20, 100, 5)), # 20,25,...,95  → 16 个
}
# 全量 = 176 组合，遗传算法只评估 pop×gen ≈ 50×10=500 次（含缓存命中会更少）

ga_opt = GeneticOptimizer(
    factory, large_grid, CONFIG,
    metric='sharpe', constraints=constraints,
    population_size=20, n_generations=10,
    mutation_rate=0.2, seed=42,
)
ga_result = ga_opt.run(klines_is)

print(f'遗传算法最优参数: {ga_result.best_params}  Sharpe={ga_result.best_score:.4f}')
print(f'实际评估次数（去重）: {len(ga_result.records)}')
ga_result.summary(top_n=5)

Output()

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

...

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

遗传算法最优参数: {'fast': 3, 'slow': 35}  Sharpe=2.2526
实际评估次数（去重）: 51


,fast,slow,total_return,annual_return,annual_vol,sharpe,sortino,max_drawdown,calmar,start,end
0,3,35,0.028568,0.135236,0.060036,2.2526,2.0369,-0.012432,10.8784,2022-01-04 09:09:00,2022-06-30 22:57:00
1,3,45,0.026691,0.125938,0.060572,2.0792,1.8026,-0.015107,8.3364,2022-01-04 09:09:00,2022-06-30 22:57:00
2,3,40,0.024317,0.114264,0.059945,1.9061,1.6547,-0.014805,7.7178,2022-01-04 09:09:00,2022-06-30 22:57:00
3,23,45,0.022837,0.107032,0.056728,1.8868,1.7929,-0.009838,10.8790,2022-01-04 09:09:00,2022-06-30 22:57:00
4,5,35,0.023546,0.110491,0.059493,1.8572,1.6873,-0.013577,8.1382,2022-01-04 09:09:00,2022-06-30 22:57:00


In [5]:
# IS vs OOS：验证最优参数在样本外是否仍然有效
from futuresquant.backtest.engine import BacktestEngine

best = ga_result.best_params
is_res  = BacktestEngine(factory(**best), CONFIG).run(klines_is)
oos_res = BacktestEngine(factory(**best), CONFIG).run(klines_oos)

print(f"参数 {best}")
print(f"  IS  Sharpe = {is_res.metrics['sharpe']:.4f}   MaxDD = {is_res.metrics['max_drawdown']:.2%}")
print(f"  OOS Sharpe = {oos_res.metrics['sharpe']:.4f}   MaxDD = {oos_res.metrics['max_drawdown']:.2%}")
ratio = oos_res.metrics['sharpe'] / is_res.metrics['sharpe'] if is_res.metrics['sharpe'] != 0 else float('nan')
print(f"  OOS/IS = {ratio:.2f}  ({'✓ 可接受' if ratio > 0.5 else '⚠ 过拟合风险'})")

Output()

Output()

参数 {'fast': 3, 'slow': 35}
  IS  Sharpe = 2.2526   MaxDD = -1.24%
  OOS Sharpe = 4.5563   MaxDD = -0.82%
  OOS/IS = 2.02  (✓ 可接受)


## 3. Walk-Forward Analysis（防过拟合核心工具）

In [6]:
wf_opt = WalkForwardOptimizer(
    factory,
    param_grid={'fast': [3, 5, 10], 'slow': [20, 30, 40]},
    config=CONFIG,
    metric='sharpe',
    constraints=constraints,
    n_splits=4,
    is_ratio=0.65,
    optimizer='grid',
)
wf_result = wf_opt.run(klines_full)

print(f'\nOOS 综合绩效:')
for k, v in wf_result.oos_metrics.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
print(f'\nIS/OOS 衰减比: {wf_result.degradation_ratio:.3f}', end='  ')
if wf_result.degradation_ratio > 0.5:
    print('✓ 参数稳健')
elif wf_result.degradation_ratio > 0:
    print('⚠ 一定程度过拟合')
else:
    print('✗ 严重过拟合')

Output()

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

...

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

Backtesting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

Output()


OOS 综合绩效:
  total_return: 0.0078
  annual_return: 0.0567
  annual_vol: 0.0620
  sharpe: 0.9156
  sortino: 0.8252
  max_drawdown: -0.0104
  calmar: 5.4816

IS/OOS 衰减比: 0.822  ✓ 参数稳健


In [7]:
# WFA 汇总表
wf_result.summary()

,窗口,IS 起始,OOS 起始,OOS 结束,IS sharpe,OOS sharpe,衰减比,param_fast,param_slow
0,1,2022-01-04,2022-03-24,2022-04-21,1.2401,-0.9260,-0.747,3,30
1,2,2022-03-08,2022-04-21,2022-05-18,0.5931,0.6087,1.026,10,40
2,3,2022-03-28,2022-05-18,2022-06-02,1.2873,0.2063,0.160,10,40
3,4,2022-04-26,2022-06-02,2022-06-22,2.8785,5.0433,1.752,3,20


In [8]:
# IS vs OOS Sharpe 对比柱状图
df_sum = wf_result.summary()
fig = go.Figure()
fig.add_trace(go.Bar(x=df_sum['窗口'].astype(str), y=df_sum['IS sharpe'],
                      name='IS Sharpe', marker_color='steelblue'))
fig.add_trace(go.Bar(x=df_sum['窗口'].astype(str), y=df_sum['OOS sharpe'],
                      name='OOS Sharpe', marker_color='#2ca02c'))
fig.add_hline(y=0, line_color='black', line_width=0.8)
fig.update_layout(title='各窗口 IS vs OOS Sharpe', barmode='group',
                   xaxis_title='窗口编号', yaxis_title='Sharpe', height=380)
fig.show()

In [14]:
# 参数稳定性：各窗口最优参数是否一致
stab = wf_result.param_stability()
print('各窗口最优参数:')
print(stab.to_string(index=False))

# 如果 slow 每窗口都相同 → 参数稳定，信号来自结构性规律而非噪音
for col in [c for c in stab.columns if c != '窗口']:
    uniq = stab[col].nunique()
    print(f'{col}: {uniq} 种取值 ({"稳定" if uniq == 1 else "变化"})')
    

各窗口最优参数:
 窗口  fast  slow
  1     3    30
  2    10    40
  3    10    40
  4     3    20
fast: 2 种取值 (变化)
slow: 3 种取值 (变化)


In [13]:
# OOS 拼接权益曲线（WFA 真实绩效）vs 全样本最优权益
oos_eq = wf_result.oos_equity.resample('1D').last().dropna()

# 全样本最优（存在前视偏差）
full_opt_res = BacktestEngine(
    factory(**grid_result.best_params), CONFIG
).run(klines_full)
full_eq = full_opt_res.account.equity_curve().resample('1D').last().dropna()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=oos_eq.index, y=oos_eq / CONFIG.initial_capital,
    name='WFA OOS（真实估计）', line=dict(color='#2ca02c', width=2)
))
fig.add_trace(go.Scatter(
    x=full_eq.index, y=full_eq / CONFIG.initial_capital,
    name='全样本最优（有前视偏差）', line=dict(color='#d62728', dash='dash')
))
fig.add_hline(y=1, line_dash='dot', line_color='gray')
fig.update_layout(
    title='WFA OOS 权益 vs 全样本最优（差距 = 过拟合程度）',
    yaxis_title='归一化权益', height=420, hovermode='x unified'
)
fig.show()

Output()